# makemore Bigram — Part 1 (count model)

Character-level bigram language model on baby names, following [Andrej Karpathy's makemore](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhqnK).

Build a **count matrix** `N`, turn it into probabilities, sample names, then measure **negative log-likelihood** and export `(xs, ys)` for Part 2.


## Load the dataset


In [ ]:
words = open('names.txt', 'r').read().splitlines()


In [ ]:
words[:10]


## Bigram counts with Python dicts

Wrap each name with start/end tokens. Here we use `<S>` and `<E>`; later we switch to `.` (index 0) to match the lecture and Part 2.


In [ ]:
freq = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        freq[bigram] = freq.get(bigram, 0) + 1


In [ ]:
freq


In [ ]:
sorted(freq.items(), key=lambda kv: -kv[1])[:3]


## Count matrix `N` (27×27)

26 letters plus one special token `.` for **start** and **end** (index 0). Each entry `N[i, j]` counts how often character `j` follows character `i`.


In [ ]:
import torch


In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)


In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}


In [ ]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        i1 = stoi[ch1]
        i2 = stoi[ch2]
        N[i1, i2] += 1


## Visualize bigram counts


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16, 16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');


## Probability and `torch.multinomial`

Row `N[0]` is the distribution after the start token. Normalize to probabilities, then **sample** an index.

The next cells build intuition: random weights → normalize → multinomial.


In [ ]:
p = N[0].float()
p = p / p.sum()
p


In [ ]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g)
ix


In [ ]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)
p = p / p.sum()
p


In [ ]:
ix = torch.multinomial(p, num_samples=20, replacement=True, generator=g).tolist()
ix


## Sample names from raw counts (no smoothing)


In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(10):
    out = []
    ix = 0
    while True:
        p = N[ix].float()
        p = p / p.sum()
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))


## Laplace smoothing and sampling

Add 1 to every count (`N + 1`), then normalize each row to get matrix `P`. This avoids zero probabilities for unseen bigrams.


In [ ]:
P = (N + 1).float()
P = P / P.sum(1, keepdim=True)


In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(10):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))


## Negative log-likelihood (loss)

**Goal:** maximize likelihood of the training names → minimize average **negative log-likelihood** (cross-entropy).

`log(a·b·c) = log(a) + log(b) + log(c)` so we sum log-probabilities over every bigram.


In [ ]:
log_likelihood = 0.0
n = 0

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'average NLL = {nll / n}')


## Training tensors for Part 2 (neural net)


In [ ]:
xs, ys = [], []
for w in words:
    ch = ['.'] + list(w) + ['.']
    for c1, c2 in zip(ch, ch[1:]):
        ix1 = stoi[c1]
        ix2 = stoi[c2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
xs.shape, ys.shape
